# 05 — Synthèse métriques & validation (EventZilla)

Ce notebook **ne réentraîne aucun modèle** : il **consolide** les résultats des notebooks **01 à 04** à partir des fichiers **`metrics_*.json`** dans `ML/models_artifacts/`, avec **objectifs**, **modèles champions**, **justifications** et **indicateurs de qualité**.

---

## Objectif global

Offrir une **vue unique** et **structurée** pour le rapport et la validation : pour chaque critère **E, C, D, F**, rappeler la **cible métier**, le **modèle retenu**, la **règle de choix** et la **qualité** des métriques enregistrées.

---

## Objectif technique

- Vérifier la présence des JSON produits par **01_E**, **02_C**, **03_D**, **04_F**.
- Construire un **tableau agrégé** (`pandas`) et une **synthèse des champions** (cibles + justification + métriques clés).
- Exporter **`ML/ML_METRICS_SUMMARY.md`** (traçabilité, relecture correcteurs).

---

## Sommaire 📑

1. **Introduction** — Rôle du notebook, chaîne **00_A → 01–04 → 05** ; critère **B** (texte) dans **`06_B_comprehension_modeles.ipynb`**.
2. **Connexion au Data Warehouse** — Diagnostic SQL (alignement SSMS).
3. **Chargement, synthèse des champions et export** — Lecture des `metrics_*.json`, affichage interprété, `DataFrame`, fichier `ML_METRICS_SUMMARY.md`.
4. **Lecture des résultats** — Comment utiliser le tableau et le rapport exporté.
5. **Bonus (barème)** — Versionnement, déploiement optionnel.

---

## Résultats attendus ✅

| Livrable | Description |
|----------|-------------|
| Synthèse par critère | Pour **E, C, D, F** : objectif, cible, champion, règle, qualité (dynamique selon les JSON). |
| Tableau `df` | Une ligne par fichier `metrics_*.json` détecté. |
| `ML_METRICS_SUMMARY.md` | JSON complets + synthèse champions + CSV — à la racine de `ML/`. |

---

## Correspondance notebooks → critères → fichiers

| Notebook | Critère | Fichier métrique (défaut) |
|----------|---------|---------------------------|
| `00_A_preparation_donnees_feature_engineering.ipynb` | **A** — Préparation | *(pas de `metrics_*.json` ; artefacts dans `ML/processed/`)* |
| `01_E_clustering_segmentation.ipynb` | **E** — Clustering | `metrics_clustering.json` |
| `02_C_classification_statut_reservation.ipynb` | **C** — Classification | `metrics_classification.json` |
| `03_D_regression_montants_KPI.ipynb` | **D** — Régression | `metrics_regression.json` |
| `04_F_series_temporelles.ipynb` | **F** — Séries temporelles | `metrics_timeseries.json` |
| `06_B_comprehension_modeles.ipynb` | **B** — Compréhension des modèles | *(documentaire ; pas de `metrics_*.json`)* |

Pour le barème **B**, utiliser **`06_B`** comme fil rouge : intuition, hypothèses, limites et justification des **paires** de modèles par tâche, en complément des métriques de ce notebook.

**Références KPI** : `docs/eventzilla/EventZilla_Dashboards_KPIs_Objectifs.md`, `ML/EventZilla_Dashboards_Improved.pdf` (champ `kpi_alignment` dans chaque JSON).

---

## Référence — critères A–F (hors sommaire)

*Bloc de référence : ne reprend pas la numérotation du sommaire.*

| Critère | Thématique ML typique |
|---------|------------------------|
| **A** | Préparation des données, feature engineering |
| **E** | Segmentation / clustering |
| **C** | Classification supervisée (statuts, funnel) |
| **D** | Régression (montants, KPI financiers) |
| **F** | Séries temporelles / prévision |

---

## 1. Introduction — rôle du notebook et chaîne d’exécution

### Rôle

Ce notebook **agrège** uniquement les métriques déjà calculées dans les notebooks **01 à 04**. Il permet de **contrôler** la cohérence des résultats, de **documenter** les modèles champions pour le rapport, et de **tracer** l’alignement avec les KPI (`kpi_alignment` dans chaque JSON).

### Chaîne recommandée

1. **`00_A_preparation_donnees_feature_engineering.ipynb`** — Matrices / features (`ML/processed/`).
2. **`01_E` … `04_F`** — Entraînement, comparaison de modèles, écriture des `metrics_*.json` sous `ML/models_artifacts/`.
3. **`05_synthese_metriques_validation.ipynb`** (ce fichier) — Synthèse finale et export `ML_METRICS_SUMMARY.md`.

---

## 2. Connexion au Data Warehouse (SQL Server / SSMS)

L’accès aux données repose sur le **même environnement** que sous **SSMS** : base **`DW_eventzella`**, via **SQLAlchemy** et **pyodbc**, authentification Windows.

La cellule de code suivante affiche serveur, base, extrait de chaîne de connexion et un test `SELECT DB_NAME()` / `SERVERPROPERTY`.

> En cas d’anomalie (service SQL, driver ODBC, variables `EVENTZILLA_SQL_*`), voir `ML/ml_paths.py`.

---

In [1]:
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "ML" / "ml_paths.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "ML" / "ml_paths.py").is_file():
    raise FileNotFoundError("Répertoire de travail attendu : racine du dépôt (dossier contenant ML/).")
sys.path.insert(0, str(REPO_ROOT))

# Connexion DW — diagnostic (même serveur / base que SSMS)
from ML.ml_paths import (
    DATABASE_DW,
    SQL_SERVER,
    SQL_DRIVER,
    SQL_PORT,
    build_windows_auth_uri,
    get_sql_engine,
    ml_sql_only,
    read_dw_sql,
    sql_engine_init_error,
)

print("=" * 62)
print(" EVENTZILLA — Connexion DW (équivalent accès SSMS)")
print("=" * 62)
print("  Serveur SQL      :", SQL_SERVER + (":" + str(SQL_PORT) if SQL_PORT else ""))
print("  Base DW cible    :", DATABASE_DW)
print("  Driver ODBC      :", SQL_DRIVER)
print("  Mode DW seul     :", ml_sql_only(), "(EVENTZILLA_ML_SQL_ONLY=1 → pas de Excel/CSV)")
try:
    _uri = build_windows_auth_uri()
    print("  Chaîne (extrait) :", (_uri[:88] + "…") if len(_uri) > 88 else _uri)
except Exception as _uerr:
    print("  Chaîne URI       : erreur", _uerr)
_eng = get_sql_engine()
if _eng is not None:
    try:
        _chk = read_dw_sql(
            "SELECT DB_NAME() AS base_active, CAST(SERVERPROPERTY('ServerName') AS NVARCHAR(128)) AS serveur",
            _eng,
        )
        print("  Test SQL         : OK — même base que sous SSMS si base_active =", DATABASE_DW)
        print(_chk.to_string(index=False))
    except Exception as _qerr:
        print("  Test SQL         : ÉCHEC —", _qerr)
else:
    print("  Engine           : ABSENT —", sql_engine_init_error() or "voir pip sqlalchemy pyodbc")
print("=" * 62)

 EVENTZILLA — Connexion DW (équivalent accès SSMS)
  Serveur SQL      : ASUSRANIM
  Base DW cible    : DW_eventzella
  Driver ODBC      : ODBC Driver 17 for SQL Server
  Mode DW seul     : True (EVENTZILLA_ML_SQL_ONLY=1 → pas de Excel/CSV)
  Chaîne (extrait) : mssql+pyodbc://@ASUSRANIM/DW_eventzella?driver=ODBC+Driver+17+for+SQL+Server&trusted_con…
  Test SQL         : OK — même base que sous SSMS si base_active = DW_eventzella
  base_active   serveur
DW_eventzella AsusRanim


## 3. Chargement des métriques, synthèse des champions et export

### Portée

- Aucun réentraînement : **lecture** des fichiers dans `ML/models_artifacts/`.
- Si un fichier manque, un **avertissement** s’affiche ; réexécutez le notebook **01–04** correspondant.

### Contenu de la cellule suivante

1. Chargement de tous les `metrics_*.json`.
2. **Tableau comparatif synthétique** (une ligne par critère **E / C / D / F**) : domaine, cible **Y**, modèle **champion**, **benchmark**, règle de choix, qualité condensée, KPI.
3. **Vue détaillée** optionnelle (colonnes larges) + **`df`** brut agrégé.
4. Écriture de **`ML/ML_METRICS_SUMMARY.md`** (tableau comparatif Markdown + détails + JSON + CSV).

---

In [2]:
# -*- coding: utf-8 -*-
# Cellule notebook 05 — chargement + tableau comparatif + export (source unique)
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "ML" / "ml_paths.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "ML" / "ml_paths.py").is_file():
    raise FileNotFoundError("Répertoire de travail attendu : racine du dépôt (dossier contenant ML/).")
sys.path.insert(0, str(REPO_ROOT))

import json
from datetime import datetime

import pandas as pd
from IPython.display import Markdown, display

from ML.ml_paths import ML_MODELS, ML_PROCESSED

EXPECTED = {
    "metrics_clustering.json": "E — Clustering",
    "metrics_classification.json": "C — Classification",
    "metrics_regression.json": "D — Régression",
    "metrics_timeseries.json": "F — Séries temporelles",
}
paths = sorted(ML_MODELS.glob("metrics_*.json"))
missing = [fn for fn in EXPECTED if not (ML_MODELS / fn).is_file()]
if missing:
    print("[0] JSON manquants (notebooks E/C/D/F non exécutés ou chemins différents) :", missing)
if not paths:
    raise FileNotFoundError(
        "Aucun fichier metrics_*.json dans ML/models_artifacts/. "
        "Ces fichiers proviennent des notebooks E, C, D, F (après la préparation 00)."
    )
rows = []
raw_by_name = {}
for p in paths:
    d = json.loads(p.read_text(encoding="utf-8"))
    d["_source_file"] = p.name
    rows.append(d)
    raw_by_name[p.name] = d
df = pd.DataFrame(rows)
print("[1] Fichiers lus :", [p.name for p in paths])


def champion_narrative(name: str, d: dict) -> dict:
    """Détail textuel (export Markdown et vue détaillée)."""
    kpi = d.get("kpi_alignment", "")
    if name == "metrics_clustering.json":
        champ = d.get("model_primary", "?")
        alt = d.get("model_secondary", "")
        k = d.get("k", "")
        sil_h = d.get("silhouette_holdout")
        db_k = d.get("davies_bouldin_kmeans")
        db_a = d.get("davies_bouldin_agg")
        return {
            "Critère": "E",
            "Objectif": "Segmentation des entités pour piloter la diversité par segment (offre / comportement).",
            "Cible / construction": f"Partition en **k = {k}** clusters à partir de features numériques standardisées.",
            "Modèle champion (principal)": f"**{champ}** (comparé à **{alt}** dans le notebook 01_E).",
            "Règle de choix / justification": "Comparaison **KMeans vs clustering hiérarchique (Ward)** : silhouette sur holdout, indice de Davies-Bouldin ; le notebook retient un **modèle principal** et un **second** pour la robustesse.",
            "Qualité (indicateurs)": f"Silhouette holdout ≈ **{sil_h}** ; Davies-Bouldin KMeans ≈ **{db_k}**, Agglomerative ≈ **{db_a}** (DB plus bas = clusters plus compacts).",
            "Lien KPI": f"`{kpi}`",
        }
    if name == "metrics_classification.json":
        champ = d.get("champion_model", "?")
        tm = d.get("test_metrics_champion", {})
        acc = tm.get("accuracy")
        f1 = tm.get("f1_weighted")
        roc = tm.get("roc_auc")
        cl = ", ".join(f"`{c}`" for c in d.get("classes", []))
        return {
            "Critère": "C",
            "Objectif": "Prédire le **statut de réservation** (funnel : annulation, confirmation, attente).",
            "Cible / construction": f"Variable catégorielle : classes {cl}.",
            "Modèle champion (principal)": f"**{champ}** (comparé à la **régression logistique** dans 02_C).",
            "Règle de choix / justification": "Métriques sur **jeu test** (accuracy, F1 pondéré, ROC-AUC) ; le champion minimise l’erreur de généralisation selon la grille critère **C**.",
            "Qualité (indicateurs)": f"Accuracy test ≈ **{acc}** ; F1 pondéré ≈ **{f1}** ; ROC-AUC ≈ **{roc}**.",
            "Lien KPI": f"`{kpi}`",
        }
    if name == "metrics_regression.json":
        champ = d.get("champion_model", "?")
        tgt = d.get("target", "")
        tr = d.get("test_champion", {})
        rmse = tr.get("rmse")
        r2 = tr.get("r2")
        mae = tr.get("mae")
        return {
            "Critère": "D",
            "Objectif": "Prédire le **montant** lié à la réservation / panier (KPI rentabilité).",
            "Cible / construction": f"Variable continue **`{tgt}`** (features tabulaires du pipeline 03_D).",
            "Modèle champion (principal)": f"**{champ}** (comparé au **Random Forest**).",
            "Règle de choix / justification": "**RMSE minimal sur le test** (avec validation croisée en amont) ; **Ridge** utile si fortes corrélations / besoin de coefficients lisibles.",
            "Qualité (indicateurs)": f"RMSE test ≈ **{rmse}** ; MAE ≈ **{mae}** ; R² ≈ **{r2}**.",
            "Lien KPI": f"`{kpi}`",
        }
    if name == "metrics_timeseries.json":
        champ = d.get("champion_model", "?")
        ser = d.get("series", "")
        expl = d.get("target_column_explained", "")
        rule = d.get("champion_rule", "")
        tc = d.get("test_champion", {})
        rmse = tc.get("rmse")
        mape = tc.get("mape")
        horiz = d.get("horizon", "")
        return {
            "Critère": "F",
            "Objectif": "Prévoir l’**évolution mensuelle** d’un agrégat DW (volume d’activité, CA ou panier moyen selon la colonne disponible).",
            "Cible / construction": f"Série **`{ser}`** — {expl}",
            "Modèle champion (principal)": f"**{champ}** (comparé à **ARIMA** dans 04_F).",
            "Règle de choix / justification": f"{rule} — holdout **{horiz}** mois.",
            "Qualité (indicateurs)": f"RMSE holdout ≈ **{rmse}** ; MAPE ≈ **{mape}** % (unité = celle de la série).",
            "Lien KPI": f"`{d.get('kpi_alignment', '')}`",
        }
    return {
        "Critère": "?",
        "Objectif": "",
        "Cible / construction": "",
        "Modèle champion (principal)": "",
        "Règle de choix / justification": "",
        "Qualité (indicateurs)": "",
        "Lien KPI": kpi,
    }


def champion_compare_row(fn: str, d):
    """Une ligne plate pour tableau comparatif (sans syntaxe Markdown)."""
    if d is None:
        return {
            "Critère": EXPECTED.get(fn, "?").split("—")[0].strip(),
            "Domaine": EXPECTED.get(fn, fn),
            "Cible (Y)": "—",
            "Champion": "—",
            "Benchmark": "—",
            "Règle de choix": "—",
            "Qualité (synthèse)": "Fichier absent",
            "KPI": "—",
            "Fichier": fn,
        }
    kpi = d.get("kpi_alignment", "")
    if fn == "metrics_clustering.json":
        sil = d.get("silhouette_holdout")
        dbk = d.get("davies_bouldin_kmeans")
        dba = d.get("davies_bouldin_agg")
        qual = (
            f"Silh.={float(sil):.4f} | DB_K={float(dbk):.3f} | DB_A={float(dba):.3f}"
            if sil is not None and dbk is not None and dba is not None
            else "—"
        )
        return {
            "Critère": "E",
            "Domaine": "Clustering",
            "Cible (Y)": f"k={d.get('k', '?')} clusters (features standardisées)",
            "Champion": str(d.get("model_primary", "?")),
            "Benchmark": str(d.get("model_secondary", "?")),
            "Règle de choix": "Silhouette (holdout) + Davies-Bouldin",
            "Qualité (synthèse)": qual,
            "KPI": kpi,
            "Fichier": fn,
        }
    if fn == "metrics_classification.json":
        tm = d.get("test_metrics_champion", {})
        return {
            "Critère": "C",
            "Domaine": "Classification",
            "Cible (Y)": "Statut réservation (multi-classes)",
            "Champion": str(d.get("champion_model", "?")),
            "Benchmark": "Régression logistique",
            "Règle de choix": "Accuracy / F1 / ROC-AUC sur test (critère C)",
            "Qualité (synthèse)": (
                f"Acc={float(tm.get('accuracy', 0)):.4f} | F1={float(tm.get('f1_weighted', 0)):.4f} | "
                f"AUC={float(tm.get('roc_auc', 0)):.4f}"
            ),
            "KPI": kpi,
            "Fichier": fn,
        }
    if fn == "metrics_regression.json":
        tr = d.get("test_champion", {})
        return {
            "Critère": "D",
            "Domaine": "Régression",
            "Cible (Y)": str(d.get("target", "?")),
            "Champion": str(d.get("champion_model", "?")),
            "Benchmark": "Random Forest",
            "Règle de choix": "RMSE minimal sur test (CV en amont)",
            "Qualité (synthèse)": (
                f"RMSE={float(tr.get('rmse', 0)):.4f} | MAE={float(tr.get('mae', 0)):.4f} | "
                f"R²={float(tr.get('r2', 0)):.6f}"
            ),
            "KPI": kpi,
            "Fichier": fn,
        }
    if fn == "metrics_timeseries.json":
        tc = d.get("test_champion", {})
        return {
            "Critère": "F",
            "Domaine": "Séries temporelles",
            "Cible (Y)": str(d.get("series", "?")),
            "Champion": str(d.get("champion_model", "?")),
            "Benchmark": "ARIMA",
            "Règle de choix": "RMSE minimal sur holdout",
            "Qualité (synthèse)": (
                f"RMSE={float(tc.get('rmse', 0)):.4f} | MAPE={float(tc.get('mape', 0)):.2f}% | "
                f"h={d.get('horizon', '?')} mois"
            ),
            "KPI": d.get("kpi_alignment", ""),
            "Fichier": fn,
        }
    return {
        "Critère": "?",
        "Domaine": fn,
        "Cible (Y)": "",
        "Champion": "",
        "Benchmark": "",
        "Règle de choix": "",
        "Qualité (synthèse)": "",
        "KPI": kpi,
        "Fichier": fn,
    }


order_fn = [
    "metrics_clustering.json",
    "metrics_classification.json",
    "metrics_regression.json",
    "metrics_timeseries.json",
]

compare_rows = [champion_compare_row(fn, raw_by_name.get(fn)) for fn in order_fn]
df_compare = pd.DataFrame(compare_rows)
cols_order = [
    "Critère",
    "Domaine",
    "Cible (Y)",
    "Champion",
    "Benchmark",
    "Règle de choix",
    "Qualité (synthèse)",
    "KPI",
    "Fichier",
]
df_compare = df_compare[[c for c in cols_order if c in df_compare.columns]]

display(
    Markdown(
        "### Tableau comparatif — synthèse des champions (critères E, C, D, F)\n\n"
        "*Une ligne par tâche ML : **Champion vs benchmark**, règle de décision et indicateurs synthétiques.*\n\n"
        "*Sources : `metrics_*.json` (réexécutez les notebooks **01–04** pour actualiser).*"
    )
)

try:
    _sty = df_compare.style.set_properties(**{"text-align": "left", "vertical-align": "top"}).set_table_styles(
        [
            {"selector": "th", "props": [("text-align", "left"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("padding", "6px 10px")]},
        ]
    )
    try:
        _sty = _sty.hide(axis="index")
    except TypeError:
        try:
            _sty = _sty.hide_index()
        except Exception:
            pass
    display(_sty)
except AttributeError as _err:
    if "jinja2" in str(_err).lower() or "style" in str(_err).lower():
        print("[info] Optionnel pour la mise en forme du tableau : pip install jinja2")
    display(df_compare)
except Exception:
    display(df_compare)

display(Markdown("#### Vue détaillée (colonnes larges)"))
summary_rows = [champion_narrative(fn, raw_by_name[fn]) for fn in order_fn if fn in raw_by_name]
if summary_rows:
    display(pd.DataFrame(summary_rows))

display(Markdown("#### Données brutes agrégées (`df` — une ligne par JSON)"))
display(df)

# --- Export ML_METRICS_SUMMARY.md ---
ts = datetime.now().strftime("%Y-%m-%d %H:%M")
out_md = ML_PROCESSED.parent / "ML_METRICS_SUMMARY.md"
parts = [
    "# EventZilla — Synthèse métriques ML et modèles champions\n\n",
    f"*Généré : {ts}*\n\n",
    "## Tableau comparatif (synthèse E, C, D, F)\n\n",
]


def _df_to_md_table(frame: pd.DataFrame) -> str:
    esc = lambda x: str(x).replace("|", "/").replace("\n", " ")
    header = "| " + " | ".join(esc(c) for c in frame.columns) + " |\n"
    sep = "| " + " | ".join("---" for _ in frame.columns) + " |\n"
    body = ""
    for _, row in frame.iterrows():
        body += "| " + " | ".join(esc(v) for v in row) + " |\n"
    return header + sep + body


parts.append(_df_to_md_table(df_compare))
parts.append("\n")

parts.append("## Détail par critère (texte)\n\n")
for fn in order_fn:
    if fn not in raw_by_name:
        parts.append(f"### `{fn}` — fichier absent\n\n")
        continue
    nar = champion_narrative(fn, raw_by_name[fn])
    parts.append(f"### Critère {nar['Critère']} — `{fn}`\n\n")
    parts.append(f"- **Objectif** : {nar['Objectif']}\n")
    parts.append(f"- **Cible / construction** : {nar['Cible / construction']}\n")
    parts.append(f"- **Modèle retenu** : {nar['Modèle champion (principal)']}\n")
    parts.append(f"- **Justification** : {nar['Règle de choix / justification']}\n")
    parts.append(f"- **Qualité** : {nar['Qualité (indicateurs)']}\n")
    parts.append(f"- **KPI** : {nar['Lien KPI']}\n\n")

parts.append("## Contenu JSON intégral par fichier\n\n")
for p in sorted(ML_MODELS.glob("metrics_*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    parts.append(f"### {p.name}\n\n")
    parts.append("```json\n")
    parts.append(json.dumps(d, indent=2, ensure_ascii=False))
    parts.append("\n```\n\n")

parts.append("## Tableau agrégé (CSV)\n\n")
parts.append("Les colonnes diffèrent selon les tâches ; le JSON ci-dessus reste la référence.\n\n```csv\n")
_drop = [c for c in df.columns if str(c).startswith("_")]
parts.append(df.drop(columns=_drop, errors="ignore").to_csv(index=False))
parts.append("\n```\n")

out_md.write_text("".join(parts), encoding="utf-8")
print("[2] Écrit :", out_md.resolve())
display(Markdown("**Export** : `ML/ML_METRICS_SUMMARY.md` (tableau comparatif + détails + JSON + CSV)."))

[1] Fichiers lus : ['metrics_classification.json', 'metrics_clustering.json', 'metrics_regression.json', 'metrics_timeseries.json']


### Tableau comparatif — synthèse des champions (critères E, C, D, F)

*Une ligne par tâche ML : **Champion vs benchmark**, règle de décision et indicateurs synthétiques.*

*Sources : `metrics_*.json` (réexécutez les notebooks **01–04** pour actualiser).*

[info] Optionnel pour la mise en forme du tableau : pip install jinja2


,Critère,Domaine,Cible (Y),Champion,Benchmark,Règle de choix,Qualité (synthèse),KPI,Fichier
0,E,Clustering,k=3 clusters (features standardisées),KMeans,AgglomerativeClustering_ward,Silhouette (holdout) + Davies-Bouldin,Silh.=0.1508 | DB_K=1.823 | DB_A=2.190,diversite_offre_segments_critere_E,metrics_clustering.json
1,C,Classification,Statut réservation (multi-classes),RandomForest,Régression logistique,Accuracy / F1 / ROC-AUC sur test (critère C),Acc=0.3360 | F1=0.3351 | AUC=0.5084,taux_acceptation_annulation_funnel_critere_C,metrics_classification.json
2,D,Régression,final_price,Ridge,Random Forest,RMSE minimal sur test (CV en amont),RMSE=1.6094 | MAE=0.7077 | R²=1.000000,panier_moyen_ca_sum_final_price,metrics_regression.json
3,F,Séries temporelles,nb_fact_rows,Holt_ExponentialSmoothing,ARIMA,RMSE minimal sur holdout,RMSE=38.0473 | MAPE=6.12% | h=3 mois,count_id_reservation_mensuel_anticipation,metrics_timeseries.json


#### Vue détaillée (colonnes larges)

,Critère,Objectif,Cible / construction,Modèle champion (principal),Règle de choix / justification,Qualité (indicateurs),Lien KPI
0,E,Segmentation des entités pour piloter la diver...,Partition en **k = 3** clusters à partir de fe...,**KMeans** (comparé à **AgglomerativeClusterin...,Comparaison **KMeans vs clustering hiérarchiqu...,Silhouette holdout ≈ **0.15084658039644186** ;...,`diversite_offre_segments_critere_E`
1,C,Prédire le **statut de réservation** (funnel :...,"Variable catégorielle : classes `cancelled`, `...",**RandomForest** (comparé à la **régression lo...,"Métriques sur **jeu test** (accuracy, F1 pondé...",Accuracy test ≈ **0.336** ; F1 pondéré ≈ **0.3...,`taux_acceptation_annulation_funnel_critere_C`
2,D,Prédire le **montant** lié à la réservation / ...,Variable continue **`final_price`** (features ...,**Ridge** (comparé au **Random Forest**).,**RMSE minimal sur le test** (avec validation ...,RMSE test ≈ **1.6093767407037995** ; MAE ≈ **0...,`panier_moyen_ca_sum_final_price`
3,F,Prévoir l’**évolution mensuelle** d’un agrégat...,Série **`nb_fact_rows`** — Volume mensuel d'ac...,**Holt_ExponentialSmoothing** (comparé à **ARI...,RMSE minimal sur le holdout; si egalite des RM...,RMSE holdout ≈ **38.04731940320474** ; MAPE ≈ ...,`count_id_reservation_mensuel_anticipation`


#### Données brutes agrégées (`df` — une ligne par JSON)

,task,criterion,champion_model,gridsearch_rf_best_params,gridsearch_lr_best_params,test_metrics_champion,test_metrics_rf,test_metrics_lr,classes,kpi_alignment,...,series,champion_rule,target_column_explained,rmse_delta_holt_minus_arima,adf_pvalue,kpss_pvalue,decomposition_period_used,test_holt,test_arima,horizon
0,classification,C,RandomForest,"{'clf__max_depth': None, 'clf__n_estimators': 80}",{'clf__C': 0.1},"{'accuracy': 0.336, 'precision_weighted': 0.33...","{'accuracy': 0.336, 'precision_weighted': 0.33...","{'accuracy': 0.336, 'precision_weighted': 0.33...","[cancelled, confirmed, pending]",taux_acceptation_annulation_funnel_critere_C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,clustering,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,diversite_offre_segments_critere_E,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,regression,D,Ridge,NaN,NaN,NaN,NaN,NaN,NaN,panier_moyen_ca_sum_final_price,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,time_series,F,Holt_ExponentialSmoothing,NaN,NaN,NaN,NaN,NaN,NaN,count_id_reservation_mensuel_anticipation,...,nb_fact_rows,RMSE minimal sur le holdout; si egalite des RM...,Volume mensuel d'activité (nombre de lignes de...,-0.602968,0.00025,0.1,12.0,"{'rmse': 38.04731940320474, 'mae': 34.54931871...","{'rmse': 38.65028712749131, 'mae': 34.29316623...",3.0


[2] Écrit : C:\Users\ranim\Downloads\PI BI NEW\ML\ML_METRICS_SUMMARY.md


**Export** : `ML/ML_METRICS_SUMMARY.md` (tableau comparatif + détails + JSON + CSV).

## 4. Lecture des résultats agrégés

- Le **DataFrame** concatène les clés de chaque JSON (certaines colonnes sont vides selon la tâche) : utile pour une vue **machine** des exports.
- La **synthèse par critère E / C / D / F** ci-dessus donne la lecture **métier** : objectif, cible, champion, justification et niveau de qualité.
- Le fichier **`ML_METRICS_SUMMARY.md`** reprend les mêmes éléments pour le **rapport** ou les correcteurs (copier-coller possible).

---

## 5. Bonus (barème) et clôture

- **Versionnement Git** : commits par phase (préparation, modèles, synthèse) pour la traçabilité.
- **Déploiement** (optionnel) : API ou application web pour servir un modèle — hors exigence des critères **A–F** dans les notebooks.

### À retenir

Ce notebook **05** matérialise la **synthèse projet** : en une lecture, vous vérifiez les **champions** des notebooks **01 à 04**, leurs **cibles**, les **règles de choix** et un **premier jugement de qualité** via les métriques exportées.

---